<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/Faiss_Semantic_260514.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu

In [ ]:
import numpy as np
import pandas as pd
import urllib.request
import faiss
import time
import torch
from sentence_transformers import SentenceTransformer

In [ ]:
df = pd.read_csv('/content/abcnews-date-text.csv',
                on_bad_lines='skip')
print(df.head())

In [ ]:
# urllib 다운로드 부분 대신 이걸로 대체
df = pd.read_csv('/content/abcnews-date-text.csv',
                 on_bad_lines='skip',
                 engine='python',
                 dtype={'publish_date': str})
data = df.headline_text.to_list()

print(data[:5])
print('총 샘플의 개수 :', len(data))

In [ ]:
model = SentenceTransformer('distilbert-base-nli-mean-tokens')

# 전체 대신 1만개만 사용 (시간 절약)
data = data[:10000]

encoded_data = model.encode(data)
print('임베딩 된 벡터 수 :', len(encoded_data))

In [ ]:
index = faiss.IndexIDMap(faiss.IndexFlatIP(768))
index.add_with_ids(encoded_data, np.array(range(0, len(data))))

faiss.write_index(index, 'abc_news')

1줄: 768차원(SBERT 출력 크기) Faiss 인덱스 생성

2줄: 임베딩된 벡터들을 인덱스에 추가

3줄: 인덱스를 abc_news 파일로 저장

In [ ]:
def search(query):
  t = time.time()
  query_vector = model.encode([query])
  k = 5
  top_k = index.search(query_vector, k)
  print('total time: {}'.format(time.time() - t))
  return [data[_id] for _id in top_k[1].tolist()[0]]

In [ ]:
query = str(input())
results = search(query)

print('results :')
for result in results:
  print('|t', result)

1. `faiss-cpu` 설치 후 import해야 faiss 사용 가능

2. GitHub 링크 막히면 Kaggle에서 데이터 직접 다운로드

3. 108만개 데이터는 너무 많으니 1만개로 줄여서 실습

4. SBERT로 텍스트를 768차원 벡터로 임베딩

5. Faiss 인덱스에 벡터 저장 후 유사 문장 시맨틱 검색 가능